# Testing New pyetm Features: Scenario Deletion & Collections CRUD

This notebook tests the newly implemented features:
- Scenario deletion (individual and bulk)
- Collections CRUD operations (create, read, update, delete)

⚠️ **Warning**: This notebook performs DELETE operations. Use with caution on production data!

## Setup

In [3]:
from pyetm import Scenario, Scenarios, Collection, Session
from pyetm.clients import BaseClient
import os

# Verify environment configuration
print("Environment check:")
print(f"ETM_API_TOKEN set: {'ETM_API_TOKEN' in os.environ}")
print(f"ENVIRONMENT: {os.environ.get('ENVIRONMENT', 'not set')}")

# Create client
client = BaseClient()
print(f"\nClient base URL: {client.session.base_url}")

Environment check:
ETM_API_TOKEN set: True
ENVIRONMENT: local

Client base URL: http://localhost:3000/api/v3


## Part 1: Test Scenario Deletion

### 1.1 Create Test Scenarios

In [4]:
# Create a few test scenarios that we'll delete
print("Creating test scenarios for deletion...\n")

test_scenarios = []
for i in range(3):
    scenario = Scenario.create(
        title=f"DELETE TEST {i+1} - Safe to Delete",
        area_code="nl2023",
        end_year=2050,
        private=True,
        client=client
    )
    test_scenarios.append(scenario)
    print(f"Created scenario {scenario.id}: {scenario.title}")

print(f"\nCreated {len(test_scenarios)} test scenarios")

Creating test scenarios for deletion...

Created scenario 181: DELETE TEST 1 - Safe to Delete
Created scenario 182: DELETE TEST 2 - Safe to Delete
Created scenario 183: DELETE TEST 3 - Safe to Delete

Created 3 test scenarios


### 1.2 Test Individual Deletion

In [5]:
# Delete the first test scenario
print("Testing individual scenario deletion...\n")

scenario_to_delete = test_scenarios[0]
print(f"Deleting scenario {scenario_to_delete.id}: {scenario_to_delete.title}")

try:
    scenario_to_delete.delete(client=client)
    print("✓ Successfully deleted scenario")
except Exception as e:
    print(f"✗ Failed to delete: {e}")

# Verify it's deleted by trying to load it
print("\nVerifying deletion...")
try:
    Scenario.load(scenario_to_delete.id, client=client)
    print("✗ Scenario still exists (unexpected)")
except Exception as e:
    print(f"✓ Scenario not found (expected): {str(e)[:100]}")

Testing individual scenario deletion...

Deleting scenario 181: DELETE TEST 1 - Safe to Delete
✓ Successfully deleted scenario

Verifying deletion...
✓ Scenario not found (expected): Scenario 181 does not exist on this ETM environment


### 1.3 Test Bulk Deletion

In [6]:
# Delete remaining test scenarios in bulk
print("Testing bulk scenario deletion...\n")

remaining_ids = [s.id for s in test_scenarios[1:]]
print(f"Deleting scenarios: {remaining_ids}")

result = Scenarios.delete_many(remaining_ids, client=client)

print(f"\nResults:")
print(f"  Successfully deleted: {result['successful']}")
print(f"  Failed to delete: {result['failed']}")

if len(result['successful']) == len(remaining_ids):
    print("\n✓ All scenarios deleted successfully")
else:
    print("\n✗ Some scenarios failed to delete")

Testing bulk scenario deletion...

Deleting scenarios: [182, 183]

Results:
  Successfully deleted: [182, 183]
  Failed to delete: []

✓ All scenarios deleted successfully


### 1.4 Test Deletion Error Handling

In [7]:
# Try to delete a non-existent scenario
print("Testing error handling for non-existent scenario...\n")

fake_id = 999999999
result = Scenarios.delete_many([fake_id], client=client)

print(f"Results for fake ID {fake_id}:")
print(f"  Successful: {result['successful']}")
print(f"  Failed: {result['failed']}")

if fake_id in result['failed']:
    print("\n✓ Error handling works correctly")
else:
    print("\n✗ Error handling issue")

Testing error handling for non-existent scenario...

Results for fake ID 999999999:
  Successful: [999999999]
  Failed: []

✗ Error handling issue


## Part 2: Test Collections CRUD

### 2.1 Create Test Scenarios for Collections

In [8]:
# Create scenarios to use in collections
print("Creating scenarios for collection testing...\n")

collection_scenarios = []
for i in range(4):
    scenario = Scenario.create(
        title=f"Collection Test Scenario {i+1}",
        area_code="nl2023",
        end_year=2050,
        private=True,
        client=client
    )
    collection_scenarios.append(scenario)
    print(f"Created scenario {scenario.id}: {scenario.title}")

scenario_ids = [s.id for s in collection_scenarios]
print(f"\nScenario IDs for collections: {scenario_ids}")

Creating scenarios for collection testing...

Created scenario 184: Collection Test Scenario 1
Created scenario 185: Collection Test Scenario 2
Created scenario 186: Collection Test Scenario 3
Created scenario 187: Collection Test Scenario 4

Scenario IDs for collections: [184, 185, 186, 187]


### 2.2 Test Collection Creation

In [12]:
# Create a simple collection
print("Testing collection creation...\n")

test_collection = Collection.create(
    title="Test Collection - Safe to Delete",
    saved_scenario_ids=scenario_ids[:3],  # Use first 3 scenarios
    interpolation=False,
    client=client
)

# print(f"✓ Created collection {test_collection.id}")
# print(f"  Title: {test_collection.title}")
print(f"  Scenarios: {test_collection.saved_scenario_ids}")
# print(f"  Interpolation: {test_collection.interpolation}")

Testing collection creation...

  Scenarios: []


### 2.3 Test Collection Loading

In [13]:
# Load single collection
print("Testing collection loading...\n")

loaded_collection = Collection.load(test_collection.id, client=client)
print(f"✓ Loaded collection {loaded_collection.id}")
print(f"  Title: {loaded_collection.title}")
print(f"  Created at: {loaded_collection.created_at}")
print(f"  Owner: {loaded_collection.owner}")

Testing collection loading...



AttributeError: 'Collection' object has no attribute 'id'

### 2.4 Test Load All Collections

In [15]:
# Load all user collections
print("Testing load all collections...\n")

all_collections = Collection.load_all(client=client)
print(f"✓ Found {len(all_collections)} collections\n")

for collection in all_collections[:5]:  # Show first 5
    print(f"  - {collection.id}: {collection.title}")

if len(all_collections) > 5:
    print(f"  ... and {len(all_collections) - 5} more")

Testing load all collections...

✓ Found 1 collections

  - 1: COLLECTION PYETM


### 2.5 Test Collection Update

In [16]:
# Update collection
print("Testing collection update...\n")

print(f"Before update:")
print(f"  Title: {test_collection.title}")
print(f"  Scenarios: {test_collection.saved_scenario_ids}")

test_collection.update(
    title="Updated Test Collection",
    saved_scenario_ids=scenario_ids,  # Add the 4th scenario
    client=client
)

print(f"\nAfter update:")
print(f"  Title: {test_collection.title}")
print(f"  Scenarios: {test_collection.saved_scenario_ids}")
print(f"\n✓ Collection updated successfully")

Testing collection update...

Before update:


AttributeError: 'Collection' object has no attribute 'title'

### 2.6 Test Collection.to_dict()

In [17]:
# Test serialization
print("Testing collection serialization...\n")

collection_dict = test_collection.to_dict()
print("✓ Collection converted to dictionary")
print(f"\nKeys: {list(collection_dict.keys())}")
print(f"\nSample data:")
for key in ['id', 'title', 'saved_scenario_ids', 'interpolation', 'created_at']:
    if key in collection_dict:
        print(f"  {key}: {collection_dict[key]}")

Testing collection serialization...

✓ Collection converted to dictionary

Keys: ['interpolation', 'area_code', 'end_year', 'version', 'owner', 'saved_scenario_ids', 'scenario_ids', 'collections_app_url', 'interpolation_params', 'discarded', 'created_at', 'updated_at']

Sample data:
  saved_scenario_ids: []
  interpolation: False
  created_at: None


/Users/louisparkes-talbot/Library/Caches/pypoetry/virtualenvs/pyetm-Rh4Np-o3-py3.12/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='version', input_value=['is missing'], input_type=list])
  return self.__pydantic_serializer__.to_python(


### 2.7 Test Interpolated Collection

In [18]:
# Create interpolated collection
print("Testing interpolated collection creation...\n")

try:
    interpolated_collection = Collection.create(
        title="Interpolated Test Collection",
        saved_scenario_ids=[scenario_ids[0]],  # Only 1 scenario for interpolation
        area_code="nl2023",
        end_year=2050,
        interpolation=True,
        client=client
    )

    print(f"✓ Created interpolated collection {interpolated_collection.id}")
    print(f"  Title: {interpolated_collection.title}")
    print(f"  Interpolation: {interpolated_collection.interpolation}")
    print(f"  Area code: {interpolated_collection.area_code}")
    print(f"  End year: {interpolated_collection.end_year}")
    if interpolated_collection.interpolation_params:
        print(f"  Interpolation params: {interpolated_collection.interpolation_params}")
except Exception as e:
    print(f"✗ Failed to create interpolated collection: {e}")
    interpolated_collection = None

Testing interpolated collection creation...

✗ Failed to create interpolated collection: 'Collection' object has no attribute 'id'


### 2.8 Test Collection Deletion

In [19]:
# Delete test collections
print("Testing collection deletion...\n")

collections_to_delete = [test_collection]
if interpolated_collection:
    collections_to_delete.append(interpolated_collection)

for collection in collections_to_delete:
    print(f"Deleting collection {collection.id}: {collection.title}")
    try:
        collection.delete(client=client)
        print(f"  ✓ Successfully deleted")
    except Exception as e:
        print(f"  ✗ Failed: {e}")

# Verify deletion
print("\nVerifying deletion...")
try:
    Collection.load(test_collection.id, client=client)
    print("✗ Collection still exists (unexpected)")
except Exception as e:
    print(f"✓ Collection not found (expected)")

Testing collection deletion...



AttributeError: 'Collection' object has no attribute 'id'

### 2.9 Test Validation - Max Scenarios

In [ ]:
# Test the 6-scenario limit
print("Testing collection validation (max 6 scenarios)...\n")

# Create 7 scenarios
print("Creating 7 scenarios...")
many_scenarios = []
for i in range(7):
    s = Scenario.create(
        title=f"Validation Test {i+1}",
        area_code="nl2023",
        end_year=2050,
        private=True,
        client=client
    )
    many_scenarios.append(s.id)

print(f"Created scenarios: {many_scenarios}")

# Try to create collection with 7 scenarios (should fail)
print("\nTrying to create collection with 7 scenarios...")
try:
    bad_collection = Collection.create(
        title="Too Many Scenarios",
        saved_scenario_ids=many_scenarios,
        client=client
    )
    print("✗ Collection created (validation not working)")
    bad_collection.delete(client=client)  # Clean up
except Exception as e:
    print(f"✓ Validation worked - got expected error: {str(e)[:100]}")

# Clean up test scenarios
print("\nCleaning up validation test scenarios...")
result = Scenarios.delete_many(many_scenarios, client=client)
print(f"Deleted {len(result['successful'])} scenarios")

## Part 3: Cleanup

In [ ]:
# Clean up all test scenarios
print("Cleaning up remaining test scenarios...\n")

if collection_scenarios:
    cleanup_ids = [s.id for s in collection_scenarios]
    result = Scenarios.delete_many(cleanup_ids, client=client)
    print(f"Deleted {len(result['successful'])} scenarios")
    if result['failed']:
        print(f"Failed to delete: {result['failed']}")

print("\n✓ Cleanup complete")

## Test Summary

Run all cells above to test:

### Scenario Deletion
- ✓ Individual scenario deletion
- ✓ Bulk scenario deletion
- ✓ Error handling for non-existent scenarios

### Collections CRUD
- ✓ Create collection
- ✓ Load single collection
- ✓ Load all collections
- ✓ Update collection
- ✓ Delete collection
- ✓ Create interpolated collection
- ✓ Serialize to dictionary
- ✓ Validation (max 6 scenarios)

All test data is cleaned up automatically at the end.